In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from scipy.ndimage import gaussian_filter
from utils import *
from cavity_correction import correct_cavity
from prefilter_correction import correct_prefilter
from ghost_correction import correct_ghost
from preprocess import preprocess

In [2]:
flat_files = sorted(glob.glob('../process/temp/*flat*.fits'))
ghost_files = sorted(glob.glob('../process/temp/*ghost*.fits'))
cavity_files = sorted(glob.glob('../process/temp/*cavity*.fits'))

print(flat_files)

['../process/temp/phi-fdt-flat_20240330T050009_V202608181831C_0463300100.fits', '../process/temp/phi-fdt-flat_20240926T114503_V202608181846C_0469260100.fits', '../process/temp/phi-fdt-flat_20241016T113003_V202608181902C_0470160100.fits', '../process/temp/phi-fdt-flat_20241027T233003_V202608181915C_0470270100.fits', '../process/temp/phi-fdt-flat_20241202T123003_V202608181931C_0472020100.fits', '../process/temp/phi-fdt-flat_20250119T210009_V202608181947C_0561190100.fits', '../process/temp/phi-fdt-flat_20250310T080009_V202608182002C_0563100100.fits', '../process/temp/phi-fdt-flat_20250915T140003_V202608182017C_0569150100.fits', '../process/temp/phi-fdt-flat_20250923T000503_V202608182032C_0569230100.fits', '../process/temp/phi-fdt-flat_20260310T040003_V202608182047C_0663100100.fits', '../process/temp/phi-fdt-flat_20260425T230003_V202608182102C_0664250100.fits']


In [3]:
dark_file = '/home/ulyanov/data/solo/phi/dark/solo_CAL1_phi-fdt-dark_20240205T033810_V202402220119C_0422051001.fits.gz'
prefilter_file = '/home/ulyanov/data/solo/phi/prefilter/phi-fdt-prefilter_20250916T023002_V202607231634C_0569160250.txt'

i = -7

cavity_file, flat_file, ghost_file = cavity_files[i], flat_files[i], ghost_files[i]

with fits.open(dark_file) as hdul:
    dark = hdul[0].data

with fits.open(cavity_file) as hdul:
    cavity = hdul[0].data

with fits.open(flat_file) as hdul:
    flat = hdul[0].data
    header = hdul[0].header

with fits.open(ghost_file) as hdul:
    ghost = hdul[0].data

#ghost = demodulate(ghost, header)
flat_ = demodulate(flat, header)
flat_[1:] /= flat_[0]
flat_ -= np.mean(flat_, axis=(-2,-1), keepdims=True)

In [4]:
plt.figure(figsize=(10,10))
plt.imshow(cavity, 'bwr', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [5]:
plt.figure(figsize=(10,10))
plt.imshow(flat_[3], 'gray', vmin=-5e-3, vmax=5e-3)
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,10))
plt.imshow(flat[0], 'gray', vmin=0.8, vmax=1.1)
plt.tight_layout()

In [5]:
#folder = '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-03-10/'
folder = '/home/ulyanov/data/solo/phi/test/'
files = sorted(glob.glob(folder + '*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-alam_20260805T010009_V202608050434C_0648050501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240101T040003_V202401090117C_0441010503.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240106T210007_V202401100517C_0441060508.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240107T000009_V202401110118C_0441070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T000009_V202402130123C_0442070501.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240207T060009_V202402130123C_0442070502.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240318T190009_V202405151841C_0443180504.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240328T060009_V202405152307C_0443281521.fits.gz',
 '/home/ulyanov/data/solo/phi/test/solo_L1_phi-fdt-ilam_20240330T044009_V202405152319C_0443301531.fits.gz',
 '/home/ulyanov/data/solo/ph

In [6]:
data, header = preprocess(files[-7],
                          dark_file=dark_file,
                          prefilter_file=prefilter_file,
                          cavity_file=cavity_file,
                          flatfield_file=flat_file,
                          ghost_file=ghost_file,
                          _realign=True,
                          _demodulate=True)

In [7]:
wv_shift = get_wv_shift(data, header)

plt.figure(figsize=(10,10))
plt.imshow(wv_shift, 'bwr', vmin=-5e-2, vmax=5e-2)
plt.tight_layout()

In [12]:
contpos = header['CONTPOS'] - 1

i = 5
j = 2

a, b = np.nanpercentile(data[i,0], 0.1), np.nanpercentile(data[i,0], 99.9)

plt.figure(figsize=(10,10))
plt.imshow(data[i,j], 'gray', vmin=-1e-3 * (b - a), vmax=1e-3 * (b - a))#, origin='lower')
plt.tight_layout()

In [9]:
plt.figure(figsize=(10,10))
plt.imshow(data[0,0])
plt.tight_layout()

In [10]:
temp = calc_continuum(data, header)

plt.figure(figsize=(10,10))
plt.imshow(temp[3], 'gray', vmin=-1e-3 * (b - a), vmax=1e-3 * (b - a))
plt.tight_layout()